In [ ]:
# Step 1: Importing Data from the Excel Sheet
from google.colab import auth
auth.authenticate_user()


import gspread
from google.auth import default
creds, _ = default()


gc = gspread.authorize(creds)


# Open the spreadsheet and get the data
worksheet = gc.open('samita1').sheet1
rows = worksheet.get_all_values()

In [ ]:
# Convert the data to a Dataset/DataFrame
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import joblib  # for saving and loading the model


# Create Dataset and assign column names
data = pd.DataFrame(rows[1:], columns=rows[0])

In [ ]:
# Step 2: Function to remove units and convert to numerics
def remove_units(value):
    if isinstance(value, str):
        for unit in ['µC/cm2', 'kV/cm', 'kV cm−1', 'mJ/cm3', 'J/cm3', 'μC/cm2', 'V/mm', 'MV/cm', '%', 'pC/N', 'pC N−1','p.m./V','µC-cm−2','pm/V']:
            value = value.replace(unit, '').strip()
        try:
            return float(value)
        except ValueError:
            return np.nan
    return value


# Apply the function to remove units
data = data.applymap(remove_units)


# Clean the column names to remove any leading or trailing spaces
data.columns = data.columns.str.strip()


# Columns we want to impute
columns_to_impute = ['remanent', 'coercive', 'dielectric constant', 'max p', 'effiency','saturation','d33','recoverable energy density']


# Step 3: Impute missing values and define features and targets
# Handle missing values in specified columns by filling with mean
imputer = SimpleImputer(strategy='mean')
data[columns_to_impute] = imputer.fit_transform(data[columns_to_impute])


# Define the feature and target variables
features = ['barium', 'calicum', 'zicronium', 'titanium', 'oxygen']
X = data[features].astype(float)
y = data[columns_to_impute]

In [ ]:
# Step 4: Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Train the model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


# Step 5: Calculate the lengths (non-null counts) for the specified columns
columns_to_count = ['barium', 'calicum', 'zicronium', 'titanium', 'oxygen', 'remanent', 'coercive',  'dielectric constant', 'max p','effiency','saturation','d33','recoverable energy density']


lengths = data[columns_to_count].count()


# Print the lengths of the specified columns
for column, length in lengths.items():
    print(f"{column}: {length}")

In [ ]:
# Step 6: Save the trained model to a file
model_filename = 'trained_model.joblib'
joblib.dump(model, model_filename)
print(f"Model saved to {model_filename}")


# Step 7: Load the model back from the file
loaded_model = joblib.load(model_filename)
print("Model loaded from file")


# Step 8: Define a new component for prediction
new_component = {
    'barium': 0.7,
    'calicum': 0.3,
    'zicronium': 0.2,
    'titanium': 0.8,
    'oxygen': 3
}

In [ ]:
# Step 9: Check if conditions are met before making predictions
if (new_component['barium'] + new_component['calicum'] == 1) and (new_component['zicronium'] + new_component['titanium'] == 1):
    new_component_df = pd.DataFrame([new_component])


    # Predict using the loaded model
    predictions = loaded_model.predict(new_component_df)
    r2 = loaded_model.score(X_test, y_test)


    # Output predicted values and R^2 score
    print(f'Predicted Remnant Value: {predictions[0][0]:.2f}')
    print(f'Predicted Coercive Value: {predictions[0][1]:.2f}')
    print(f'Predicted Dielectric Constant Value: {predictions[0][2]:.2f}')
    print(f'Predicted Max Polarization Value: {predictions[0][3]:.2f}')
    print(f'Predicted Efficiency Value: {predictions[0][4]:.2f}')
    print(f'Predicted saturation Value: {predictions[0][5]:.2f}')
    print(f'Predicted d33 Value: {predictions[0][6]:.2f}')
    print(f'Predicted recoverable energy density Value: {predictions[0][7]:.2f}')
    print(f'R^2 Score: {r2:.2f}')
else:
    print("The conditions are not met. The sum of 'barium' and 'calicum' must be 1, and the sum of 'zicronium' and 'titanium' must be 1.")